# Training a Reasoning Model with GRPO using Unsloth

This notebook demonstrates Group Relative Policy Optimization (GRPO) for training reasoning models using Unsloth. GRPO is a reinforcement learning technique designed for training models on reasoning tasks where the model generates its own solutions.

## What we'll cover:
- Installing Unsloth and dependencies
- Loading a model for GRPO training
- Using a dataset with problems (no predetermined answers)
- Training with GRPO where the model generates multiple solutions
- The best solutions are used as positive examples
- Testing the reasoning capabilities

## About GRPO (Group Relative Policy Optimization):
GRPO is an RL method that:
- Generates multiple candidate solutions for each problem
- Evaluates which solutions are correct/better
- Uses the best solutions to reinforce learning
- Particularly effective for math, coding, and reasoning tasks
- Similar to how DeepSeek R1 and other reasoning models are trained

## Key Difference from DPO:
- **DPO**: Requires pre-labeled chosen/rejected pairs
- **GRPO**: Model generates its own solutions, then learns from the best ones
- **GRPO**: Better for reasoning tasks where multiple solution paths exist

## Dataset Format:
GRPO requires problems with verifiable solutions:
- `problem` or `question`: The problem statement
- `answer` (optional): Ground truth for verification
- Model generates solutions and learns from successful ones

In [8]:
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dsgtt519/unsloth_f818725cdce64add96bb0b78f27d162e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dsgtt519/unsloth_f818725cdce64add96bb0b78f27d162e
  Resolved https://github.com/unslothai/unsloth.git to commit 341ce85864d191e4a6b7c447b9167c1faf5e20d3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 118.4 MB/s eta 0:00:00

## Import Required Libraries

Import the necessary libraries for GRPO training. We'll use:
- `FastLanguageModel` from Unsloth for model loading
- `GRPOTrainer` and `GRPOConfig` from TRL for Group Relative Policy Optimization
- Standard PyTorch and datasets libraries
- Tools for evaluating mathematical/reasoning outputs

In [9]:
# Import necessary libraries
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import re

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB


## Model Configuration for GRPO Training

For GRPO (Group Relative Policy Optimization), we'll use LoRA with 4-bit quantization for memory efficiency.

Configuration parameters:
- `max_seq_length`: Maximum sequence length (2048 tokens)
- `dtype`: Data type (None for auto-detection)
- `load_in_4bit`: True for memory-efficient 4-bit quantization
- `model_name`: We'll use SmolLM2 135M for consistency

GRPO works well with LoRA because:
- The model needs to generate multiple candidate solutions per problem
- Memory efficiency is important when generating multiple outputs
- LoRA allows efficient training on reasoning tasks

In [10]:
# Configuration parameters for GRPO training
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype
load_in_4bit = True  # Use 4-bit quantization for efficiency

# Model selection - using SmolLM2 135M
model_name = "unsloth/SmolLM2-135M-Instruct"

print(f"Model: {model_name}")
print(f"Max Sequence Length: {max_seq_length}")
print(f"4-bit Quantization: {load_in_4bit}")
print("GRPO training for reasoning tasks")

Model: unsloth/SmolLM2-135M-Instruct
Max Sequence Length: 2048
4-bit Quantization: True
GRPO training for reasoning tasks


## Load Model and Configure LoRA Adapters for GRPO

We load the model with 4-bit quantization and add LoRA adapters for GRPO training.

For GRPO training, the model will:
- Generate multiple solution attempts for each problem
- Learn from the best/correct solutions
- Improve its reasoning capabilities through reinforcement learning

The LoRA configuration is similar to previous methods, but the training process involves generating and evaluating multiple candidate solutions.

In [11]:
# Load model and tokenizer with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model loaded successfully with LoRA adapters for GRPO!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
trainable_percentage = 100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())
print(f"Trainable percentage: {trainable_percentage:.2f}%")

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.3 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


Model loaded successfully with LoRA adapters for GRPO!
Total parameters: 86,315,904
Trainable parameters: 4,884,480
Trainable percentage: 5.66%


## Load GRPO Training Dataset

For GRPO training, we need a dataset with problems that have verifiable solutions. We'll use a math problem dataset where:
- The model can generate multiple solution attempts
- Solutions can be evaluated for correctness
- The model learns from successful reasoning paths

Popular datasets for GRPO/reasoning training:
- `gsm8k`: Grade school math problems
- `MATH`: Mathematical reasoning problems
- `openai/prm800k`: Process-supervised math dataset
- Simple arithmetic problems for quick demonstration

We'll use GSM8K (a subset) which contains grade school math word problems with numerical answers that can be automatically verified.

In [12]:
# Load a reasoning/math dataset for GRPO training
# Using GSM8K - grade school math problems
dataset = load_dataset("gsm8k", "main", split="train")

# Take a smaller subset for quick training
dataset = dataset.select(range(min(500, len(dataset))))

print(f"Dataset loaded successfully!")
print(f"Number of examples: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print("\nFirst example:")
print(f"Question: {dataset[0]['question']}")
print(f"\nAnswer: {dataset[0]['answer']}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully!
Number of examples: 500
Dataset columns: ['question', 'answer']

First example:
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## Format Dataset and Define Reward Function for GRPO

For GRPO training, we need:
1. **Formatted prompts**: Clear problem statements for the model
2. **Reward function**: A way to evaluate if generated solutions are correct

The reward function:
- Extracts the final answer from model-generated text
- Compares it with the ground truth answer
- Returns a reward (typically 1.0 for correct, 0.0 for incorrect)

This is crucial for GRPO as the model learns from the rewards it receives for its generated solutions.

In [13]:
# Function to extract numerical answer from text
def extract_answer(text):
    """
    Extract the final numerical answer from model output.
    Looks for patterns like 'answer is X' or just numbers at the end.
    """
    # Try to find explicit answer patterns
    patterns = [
        r'####\s*([0-9,\.]+)',  # GSM8K format
        r'answer is:?\s*([0-9,\.]+)',
        r'=\s*([0-9,\.]+)\s*$',
        r'([0-9,\.]+)\s*$'  # Number at end
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).replace(',', '')
    return None

# Format dataset for GRPO
def format_grpo_dataset(examples):
    """
    Format dataset for GRPO training.
    Create prompts and extract ground truth answers.
    IMPORTANT: Column must be named "prompt" not "query"
    """
    prompts = []
    answers = []

    for question, answer in zip(examples['question'], examples['answer']):
        # Create a clear prompt
        prompt = f"""Solve this math problem step by step and provide the final numerical answer.

Problem: {question}

Solution:"""
        prompts.append(prompt)

        # Extract ground truth answer
        ground_truth = extract_answer(answer)
        answers.append(ground_truth)

    return {
        'prompt': prompts,  # Must be "prompt" not "query"
        'answer': answers
    }

# Apply formatting
formatted_dataset = dataset.map(
    format_grpo_dataset,
    batched=True,
    remove_columns=dataset.column_names
)

print("Dataset formatted for GRPO training!")
print(f"Columns: {formatted_dataset.column_names}")
print("\nExample:")
print(f"Prompt: {formatted_dataset[0]['prompt'][:300]}...")
print(f"\nGround truth answer: {formatted_dataset[0]['answer']}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset formatted for GRPO training!
Columns: ['answer', 'prompt']

Example:
Prompt: Solve this math problem step by step and provide the final numerical answer.

Problem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Solution:...

Ground truth answer: 72


## Create Reward Function for GRPO

The reward function is critical for GRPO. It:
- Takes the model's generated responses
- Evaluates their correctness
- Returns rewards (scores) for each response

In GRPO:
- The model generates multiple candidate solutions
- Each solution is evaluated with the reward function
- High-reward solutions are used to reinforce learning
- Low-reward solutions teach the model what to avoid

For math problems:
- Reward = 1.0 if the answer is correct
- Reward = 0.0 if the answer is incorrect
- This binary reward signal guides the model to generate correct solutions

In [14]:
# Define the reward function for GRPO
def reward_function(prompts, completions, **kwargs):
    """
    Reward function for GRPO training.

    Args:
        prompts: List of prompt strings
        completions: List of completion strings (generated responses)
        **kwargs: Additional arguments including 'answer' from dataset

    Returns:
        List of rewards (1.0 for correct, 0.0 for incorrect)
    """
    rewards = []

    # Get ground truth answers from kwargs
    ground_truths = kwargs.get('answer', [None] * len(completions))

    for completion, ground_truth in zip(completions, ground_truths):
        # Extract predicted answer from model output
        predicted = extract_answer(completion)

        # Compare with ground truth
        if predicted is not None and ground_truth is not None:
            try:
                # Convert to float for comparison
                pred_num = float(predicted)
                truth_num = float(ground_truth)

                # Check if answers match (with small tolerance for floating point)
                if abs(pred_num - truth_num) < 0.01:
                    rewards.append(1.0)  # Correct answer
                else:
                    rewards.append(0.0)  # Incorrect answer
            except:
                rewards.append(0.0)  # Parsing error
        else:
            rewards.append(0.0)  # Couldn't extract answer

    return rewards

print("Reward function defined!")
print("The function signature matches GRPO requirements:")
print("  - Takes: prompts, completions, **kwargs")
print("  - Returns: list of rewards (1.0 for correct, 0.0 for incorrect)")

Reward function defined!
The function signature matches GRPO requirements:
  - Takes: prompts, completions, **kwargs
  - Returns: list of rewards (1.0 for correct, 0.0 for incorrect)


## Setup GRPO Training Configuration

GRPO uses `GRPOConfig` for configuration with specific parameters for reinforcement learning.

Key GRPO-specific parameters:
- `num_sample_generations`: Number of solution attempts per problem (typically 4-8)
- `max_new_tokens`: Maximum length for generated solutions
- `temperature`: Sampling temperature for diversity (higher = more diverse solutions)
- `kl_coef`: KL divergence coefficient to prevent model from diverging too far from base model

Standard training parameters:
- `per_device_train_batch_size`: Batch size
- `gradient_accumulation_steps`: Gradient accumulation
- `learning_rate`: Learning rate (typically lower for RL)
- `max_steps`: Total training steps

GRPO generates multiple solutions per problem, evaluates them, and learns from the best ones.

In [18]:
# Setup GRPO training configuration
grpo_config = GRPOConfig(
    # Training parameters
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # GRPO-specific parameters
    num_generations=6,  # Number of solutions generated per problem
    max_prompt_length=256,  # Max length for prompts
    max_completion_length=max_seq_length - 256,  # Max length for completions

    # Standard parameters
    warmup_ratio=0.1,
    max_steps=5,  # Training steps (reduced for demo)
    logging_steps=1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    adam_beta1=0.9,
    adam_beta2=0.99,
    max_grad_norm=0.1,
    seed=3407,
    output_dir="outputs_grpo",
    report_to="none",
)

print("GRPO training configuration set!")
print(f"Generations per problem: {grpo_config.num_generations}")
print(f"Max prompt length: {grpo_config.max_prompt_length}")
print(f"Max completion length: {grpo_config.max_completion_length}")
print(f"Learning rate: {grpo_config.learning_rate}")
print(f"Total training steps: {grpo_config.max_steps}")

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6
GRPO training configuration set!
Generations per problem: 6
Max prompt length: 256
Max completion length: 1792
Learning rate: 5e-06
Total training steps: 5


## Initialize the GRPO Trainer

The `GRPOTrainer` handles the Group Relative Policy Optimization training process.

Key aspects:
- Takes the model with LoRA adapters
- Uses the formatted dataset with queries and answers
- Requires a reward function (or list of reward functions) to evaluate generated solutions
- Generates multiple solutions per problem and learns from the best ones

Important parameters:
- `model`: The model with LoRA adapters
- `processing_class`: The tokenizer (can also use `tokenizer` parameter)
- `reward_funcs`: List of reward functions to evaluate solutions
- `args`: The GRPOConfig we just created
- `train_dataset`: The formatted dataset with "query" column

The trainer will generate `num_generations` solutions for each problem and use the reward function to determine which are best.

In [19]:
# Initialize the GRPO Trainer
grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,  # Can also use tokenizer=tokenizer
    reward_funcs=[reward_function],  # Our custom reward function
    args=grpo_config,
    train_dataset=formatted_dataset,
)

print("GRPO Trainer initialized successfully!")
print(f"Training dataset size: {len(grpo_trainer.train_dataset)}")
print(f"The model will generate {grpo_config.num_generations} solutions per problem")
print(f"Solutions will be evaluated using the reward function")
print(f"The model learns from high-reward (correct) solutions")

GRPO Trainer initialized successfully!
Training dataset size: 500
The model will generate 6 solutions per problem
Solutions will be evaluated using the reward function
The model learns from high-reward (correct) solutions


## Train the Model with GRPO

Now we start the GRPO training process. The trainer will:
- For each problem, generate multiple candidate solutions (6 solutions per problem)
- Evaluate each solution using the reward function
- Compute advantages based on relative rewards within the group
- Update model parameters to increase probability of high-reward solutions
- Log training metrics including rewards and losses

Expected behavior:
- Initial rewards may be low (model generates incorrect solutions)
- Over time, rewards should increase as the model learns
- The model develops reasoning capabilities by trial and error
- Training may take longer than supervised fine-tuning due to generation

Watch the reward column in the training logs - it should increase over time as the model improves!

Note: GRPO training can take significant time due to generating multiple solutions per batch.

In [20]:
# Start GRPO training
print("Starting GRPO training...")
print("The model will generate multiple solutions and learn from the best ones.")
print("Watch for the reward values to increase over time!")
print("-" * 50)

trainer_stats = grpo_trainer.train()

print("\n" + "=" * 50)
print("GRPO Training completed!")
print("=" * 50)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print("\nThe model has learned reasoning capabilities through reinforcement learning!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting GRPO training...
The model will generate multiple solutions and learn from the best ones.
Watch for the reward values to increase over time!
--------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 4 x 1) = 24
 "-____-"     Trainable parameters = 4,884,480 of 139,400,064 (3.50% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / reward_function / mean,rewards / reward_function / std
1,0.000000,0.000000,0.000000,983.166687,1.000000,1792.000000,0.500000,174.333344,1.000000,1158.000000,0,0,0,0,0,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,1014.791687,1.000000,1792.000000,0.500000,237.583344,1.000000,1042.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,560.125000,1.000000,1792.000000,0.250000,149.500000,1.000000,630.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,926.000000,1.000000,1792.000000,0.416667,307.428589,1.000000,1431.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,634.500000,1.000000,1792.000000,0.250000,248.666672,1.000000,995.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.000000,0.000000



GRPO Training completed!
Training time: 613.00 seconds
Training samples per second: 0.20

The model has learned reasoning capabilities through reinforcement learning!


## Test the GRPO Fine-tuned Reasoning Model

After GRPO training, the model should have improved reasoning capabilities. Let's test it with a math problem to see if it can:
- Break down the problem step by step
- Show its reasoning process
- Arrive at the correct answer

Compare this with the base model's performance to see the improvement from GRPO training.

In [21]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Create a test math problem
test_prompt = """Solve this math problem step by step and provide the final numerical answer.

Problem: A baker made 24 cupcakes. She sold 3/4 of them in the morning. How many cupcakes does she have left?

Solution:"""

# Tokenize the input
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("GRPO-trained reasoning model response:")
print("=" * 50)
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)
print("=" * 50)
print("\nThe model should show step-by-step reasoning and arrive at the answer: 6 cupcakes")

GRPO-trained reasoning model response:

Let's start by setting up the equation: 24 cupcakes = 3/4 * 24.

Now, we can divide both sides of the equation by 24 to solve for the value of "x": 3/4 * 24 = 24 * 24 / 24.

Next, we can simplify the equation by dividing both sides by 24: 128 = 24 * 24 / 24.

Now, we can divide both sides of the equation by 24 to solve for "x": 128 / 24 = 24 / 24.

To solve for "x", we can divide both sides of the equation by 24: 128 / 24 = 24 / 24.

Now, we can divide both sides of the equation by 24 to solve for "x": 128 / 24 = 24 / 24.

To solve for "x", we can divide both sides of the equation by 24: 24 / 24 = 24 /

The model should show step-by-step reasoning and arrive at the answer: 6 cupcakes
